# W1-D3-DAT-1: LLM serving spike on Kaggle (Ollama + Cloudflare quick tunnel)

Runbook: `docs/runbooks/llm-serving.md`. Before running:

1. **Settings → Accelerator: GPU T4 x2** (or P100) and **Internet: On** (needs a phone-verified account).
2. Run all cells top to bottom. Do not paste any key or token into this notebook.
3. The tunnel URL printed in step 6 is temporary and **unauthenticated**: share it only with the team, never commit it, and stop the session when done.
4. Copy the summary tables of steps 5 and 7 into the runbook. Clear outputs before committing (`nbstripout`).

## 1. Hardware

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc && free -g | head -2

## 2. Install Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## 3. Start the server (all interfaces, keep models loaded)

In [ ]:
import collections
import os
import subprocess
import time
import urllib.request


def start_server() -> subprocess.Popen:
    env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1")
    proc = subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=open("ollama.log", "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,  # not affected by signals sent to the notebook kernel
    )
    for _ in range(60):
        try:
            print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
            return proc
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama did not start; see ollama.log")


server = start_server()

## 4. Pull the candidate models (timed)

In [ ]:
MODELS = ["qwen2.5:3b", "llama3.2:3b", "qwen2.5:7b"]
for m in MODELS:
    t = time.perf_counter()
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
    print(f"{m}: pulled in {time.perf_counter() - t:.0f}s")
!ollama list

## 5. Benchmark on the server itself (no tunnel)

`bench_llm.py` is the same file as `notebooks/llm-serving/bench_llm.py` in the repository.

In [ ]:
%%writefile bench_llm.py
"""Measure time to first token and generation speed of an OpenAI-compatible endpoint.

Task W1-D3-DAT-1 (LLM serving spike). Used on Kaggle against the local Ollama server and
from a laptop through the tunnel, with the same settings, so the numbers are comparable.

Configuration follows docs/contracts/llm-client.md: LLM_BASE_URL, LLM_API_KEY, LLM_MODEL.
The API key is never printed or written to the results file.

Example:
    LLM_BASE_URL=http://localhost:11434/v1 python bench_llm.py \
        --models qwen2.5:3b,llama3.2:3b --runs 5 --label kaggle-t4-local
"""

from __future__ import annotations

import argparse
import json
import os
import platform
import statistics
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

from openai import OpenAI

PROMPTS = {
    "en": (
        "Explain in about 150 words what retrieval-augmented generation is and why it "
        "can reduce hallucinations in question answering."
    ),
    "vi": (
        "Hãy giải thích bằng tiếng Việt, khoảng 150 từ, retrieval-augmented generation "
        "là gì và vì sao nó có thể giảm hallucination khi trả lời câu hỏi."
    ),
}


@dataclass
class RunResult:
    label: str
    model: str
    prompt_id: str
    run: int
    warmup: bool
    ttft_s: float | None
    total_s: float
    completion_tokens: int
    token_source: str  # "usage" (reported by the server) or "chunks" (count of stream chunks)
    decode_tps: float | None  # tokens after the first one / time after the first token
    e2e_tps: float  # tokens / total time
    temperature: float
    max_tokens: int
    started_at: str
    error: str | None = None


def run_once(
    client: OpenAI,
    model: str,
    prompt_id: str,
    run: int,
    warmup: bool,
    args: argparse.Namespace,
) -> RunResult:
    started_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    t0 = time.perf_counter()
    t_first: float | None = None
    chunks = 0
    usage_tokens: int | None = None
    error: str | None = None
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": PROMPTS[prompt_id]}],
            max_tokens=args.max_tokens,
            temperature=args.temperature,
            stream=True,
            stream_options={"include_usage": True},
        )
        for event in stream:
            if event.usage is not None:
                usage_tokens = event.usage.completion_tokens
            if event.choices and event.choices[0].delta.content:
                if t_first is None:
                    t_first = time.perf_counter()
                chunks += 1
    except Exception as exc:  # noqa: BLE001 - a spike records every failure and continues
        error = f"{type(exc).__name__}: {exc}"
    t_end = time.perf_counter()

    tokens = usage_tokens if usage_tokens is not None else chunks
    total = t_end - t0
    ttft = None if t_first is None else t_first - t0
    decode_tps = None
    if t_first is not None and tokens > 1 and t_end > t_first:
        decode_tps = (tokens - 1) / (t_end - t_first)
    return RunResult(
        label=args.label,
        model=model,
        prompt_id=prompt_id,
        run=run,
        warmup=warmup,
        ttft_s=ttft,
        total_s=total,
        completion_tokens=tokens,
        token_source="usage" if usage_tokens is not None else "chunks",
        decode_tps=decode_tps,
        e2e_tps=tokens / total if total > 0 else 0.0,
        temperature=args.temperature,
        max_tokens=args.max_tokens,
        started_at=started_at,
        error=error,
    )


def summarise(results: list[RunResult]) -> str:
    lines = [
        "| label | model | prompt | n | TTFT median (min-max) s | decode tok/s median (min-max)"
        " | tokens median | errors |",
        "|---|---|---|---|---|---|---|---|",
    ]
    keys = sorted({(r.label, r.model, r.prompt_id) for r in results if not r.warmup})
    for label, model, prompt_id in keys:
        group = [
            r
            for r in results
            if not r.warmup and (r.label, r.model, r.prompt_id) == (label, model, prompt_id)
        ]
        ok = [r for r in group if r.error is None]
        ttft = [r.ttft_s for r in ok if r.ttft_s is not None]
        tps = [r.decode_tps for r in ok if r.decode_tps is not None]
        toks = [r.completion_tokens for r in ok]

        def fmt(xs: list[float], digits: int) -> str:
            if not xs:
                return "n/a"
            return f"{statistics.median(xs):.{digits}f} ({min(xs):.{digits}f}-{max(xs):.{digits}f})"

        lines.append(
            f"| {label} | {model} | {prompt_id} | {len(ok)} | {fmt(ttft, 2)} | {fmt(tps, 1)}"
            f" | {statistics.median(toks) if toks else 'n/a'} | {len(group) - len(ok)} |"
        )
    return "\n".join(lines)


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--models", default=os.environ.get("LLM_MODEL", ""))
    parser.add_argument("--runs", type=int, default=5, help="measured runs per model and prompt")
    parser.add_argument("--prompts", default="en,vi", help=f"subset of {sorted(PROMPTS)}")
    parser.add_argument("--max-tokens", type=int, default=200)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--timeout", type=float, default=300.0)
    parser.add_argument("--label", default=platform.node() or "unknown")
    parser.add_argument("--out", default="bench_results.jsonl", help="JSONL file (appended)")
    args = parser.parse_args(argv)

    base_url = os.environ.get("LLM_BASE_URL")
    models = [m.strip() for m in args.models.split(",") if m.strip()]
    prompt_ids = [p.strip() for p in args.prompts.split(",") if p.strip()]
    if not base_url or not models:
        print("Set LLM_BASE_URL and --models (or LLM_MODEL).", file=sys.stderr)
        return 2
    unknown = set(prompt_ids) - set(PROMPTS)
    if unknown:
        print(f"Unknown prompt ids: {sorted(unknown)}", file=sys.stderr)
        return 2

    client = OpenAI(
        base_url=base_url,
        api_key=os.environ.get("LLM_API_KEY", "not-needed"),
        timeout=args.timeout,
        max_retries=0,  # measure the endpoint as it is; retries would hide failures
    )
    print(f"label={args.label} models={models} runs={args.runs} max_tokens={args.max_tokens}")

    results: list[RunResult] = []
    with open(args.out, "a", encoding="utf-8") as fh:
        for model in models:
            # One warm-up call loads the model into memory; it is recorded but not summarised.
            plan = [(prompt_ids[0], 0, True)]
            plan += [(p, i, False) for i in range(1, args.runs + 1) for p in prompt_ids]
            for prompt_id, run, warmup in plan:
                r = run_once(client, model, prompt_id, run, warmup, args)
                results.append(r)
                fh.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")
                fh.flush()
                tag = "warmup" if warmup else f"run {run}"
                ttft = "n/a" if r.ttft_s is None else f"{r.ttft_s:.2f}s"
                tps = "n/a" if r.decode_tps is None else f"{r.decode_tps:.1f} tok/s"
                print(
                    f"{model} {prompt_id} {tag}: ttft={ttft} decode={tps} "
                    f"tokens={r.completion_tokens} total={r.total_s:.2f}s"
                    + (f" ERROR {r.error}" if r.error else "")
                )

    print()
    print(summarise(results))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
!pip -q install "openai>=1.40"
%env LLM_BASE_URL=http://localhost:11434/v1
GPU_Q = (
    "timestamp,index,temperature.gpu,clocks.sm,"
    "clocks_throttle_reasons.active,power.draw,memory.used"
)
gpu_log = subprocess.Popen(
    ["nvidia-smi", f"--query-gpu={GPU_Q}", "--format=csv", "-l", "2"],
    stdout=open("gpu_during_bench.csv", "w"),
)
!python bench_llm.py --models {",".join(MODELS)} --runs 5 \
    --label kaggle-local --out bench_kaggle.jsonl
gpu_log.terminate()
!nvidia-smi --query-gpu=memory.used --format=csv
rows = [line.split(", ") for line in open("gpu_during_bench.csv").read().splitlines()[1:]]
temps = [int(r[2]) for r in rows if len(r) > 2 and r[2].isdigit()]
# Reason bits: 0x1 idle (not a slowdown), 0x4 SW power cap, 0x20 SW thermal,
# 0x40 HW thermal, 0x80 HW power brake
busy = [r for r in rows if len(r) > 6 and r[6] != "0 MiB"]
reasons = collections.Counter(r[4] for r in busy if int(r[4], 16) & ~0x1)
print(f"GPU samples: {len(rows)} ({len(busy)} with a model loaded),",
      f"max temp: {max(temps, default=None)} C")
print("slowdown reasons while loaded (code: samples):", dict(reasons) or "none")

## 6. Open the tunnel

In [ ]:
import re

CF = "https://github.com/cloudflare/cloudflared/releases/latest/download"
!wget -q -O cloudflared {CF}/cloudflared-linux-amd64 && chmod +x cloudflared
offset = os.path.getsize("tunnel.log") if os.path.exists("tunnel.log") else 0
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:11434"],
    stdout=open("tunnel.log", "a"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
url = None
for _ in range(60):
    with open("tunnel.log") as fh:
        fh.seek(offset)  # only this run's output
        found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", fh.read())
    if found:
        url = found[0]
        break
    time.sleep(1)
print("LLM_BASE_URL =", f"{url}/v1" if url else "not found, see tunnel.log")

## 7. From the laptop

```bash
pip install "openai>=1.40"
export LLM_BASE_URL=<url from step 6>/v1   # PowerShell: $env:LLM_BASE_URL="<url>/v1"
python notebooks/llm-serving/bench_llm.py --models qwen2.5:3b,llama3.2:3b,qwen2.5:7b --runs 5 --label laptop-via-tunnel --out bench_tunnel.jsonl
```

## 8. Keep the session alive while testing (stop this cell to end)

In [ ]:
# Watchdog: restarts Ollama if it exits (the tunnel URL stays valid); reports a dead tunnel.
start = time.time()
try:
    while True:
        time.sleep(60)
        minutes = (time.time() - start) / 60
        if server.poll() is not None:
            print(f"{minutes:.0f} min: Ollama exited with code {server.returncode}; restarting")
            !tail -5 ollama.log
            server = start_server()
        if "tunnel" not in globals():
            print(f"{minutes:.0f} min: no tunnel yet; run step 6 first")
            break
        if tunnel.poll() is not None:
            print(f"{minutes:.0f} min: tunnel exited with code {tunnel.returncode}; rerun step 6")
            break
        print(f"{minutes:.0f} min: server and tunnel alive", flush=True)
except KeyboardInterrupt:
    print("watchdog stopped; server and tunnel keep running")

## 9. Clean up (manual)

Set `CLEANUP = True` and run this cell only when you are done; Run All skips it.

In [ ]:
CLEANUP = False
if CLEANUP:
    tunnel.terminate()
    server.terminate()
    print(open("ollama.log").read()[-2000:])